<a href="https://colab.research.google.com/github/Akechi1412/Phishing-Website-Detection/blob/main/app/notebooks/openphish_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/Github
%cd Phishing-Website-Detection/app
!git config --global user.email 'nguyenphong10042002@gmail.com'
!git config --global user.name 'Akechi1412'
!git fetch origin
!git reset --hard origin/main

/content/drive/MyDrive/Github
/content/drive/MyDrive/Github/Phishing-Website-Detection/app
Updating files: 100% (80/80), done.
HEAD is now at 797ad89 Update openphish.txt


In [3]:
phishing_urls = []

with open('data/openphish.txt', "r", encoding='utf-8') as file:
    for line in file:
        phishing_urls.append(line.strip())

print(f'Initial number of samples: {len(phishing_urls)}')

Initial number of samples: 500


In [4]:
!pip install aiohttp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.9/241.9 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.6/124.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.1/205.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 24.4 MB/s eta 0:00:00


In [5]:
import asyncio
import aiohttp
import pickle
import numpy as np
import tensorflow as tf
from tensorflow import keras
from utils.data_preprocessing import vectorize_url, parse_html, create_graph
from utils.data_preprocessing import create_graph_adjacency, create_graph_feature

In [6]:
async def fetch_url(url, timeout=5):
    url = url.strip()
    if not url.startswith(('http://', 'https://')):
        url = 'http://' + url
    elif url.startswith('https://'):
        url = url.replace('https://', 'http://')
    async with aiohttp.ClientSession() as session:
        async with session.get(url, timeout=timeout) as response:
            response.raise_for_status()
            html = None
            content_type = response.headers.get('Content-Type', '')
            if 'text/html' in content_type.lower():
                html = await response.text()

            return {'url': str(response.url), 'html': html}

In [7]:
url_inputs = []
adjacency_inputs = []
feature_inputs = []
labels = []

max_words = 50
max_nodes = 600
feature_dim = 3

for url in phishing_urls:
    try:
        loop = asyncio.get_running_loop()
        result = await loop.create_task(fetch_url(url, timeout=10))
        if result['html'] is None:
            print(f'Error fetching URL: {url}')
            continue
    except Exception as e:
        print(f'Error fetching URL: {url}')
        print(f'Error message: {e}')
        continue

    with open('models/dictionary.pkl', 'rb') as f:
        dictionary = pickle.load(f)
    url_input = vectorize_url(result['url'],
                            dictionary=dictionary,
                            max_words=max_words)
    html_dom = parse_html(result['html'])
    html_graph = create_graph(html_dom)
    adjacency_input = create_graph_adjacency(html_graph, max_nodes=max_nodes)
    feature_input = create_graph_feature(html_graph, max_nodes=max_nodes)

    url_inputs.append(url_input)
    adjacency_inputs.append(adjacency_input)
    feature_inputs.append(feature_input)
    labels.append(1)

print(f'Number of samples after fetching HTML: {len(labels)}')

Error fetching URL: http://myfiles3546.pages.dev/
Error message: 403, message='Forbidden', url='http://myfiles3546.pages.dev/'
Error fetching URL: https://steamcomnurnity.ru/Id/7656119798766477
Error message: 404, message='Not Found', url='http://steamcomnurnity.ru/Id/7656119798766477'
Error fetching URL: https://evmdebug.pages.dev/
Error message: 403, message='Forbidden', url='http://evmdebug.pages.dev/'
Error fetching URL: https://mdyfsvb.kwusg.my.id/verify.php
Error message: Cannot connect to host mdyfsvb.kwusg.my.id:80 ssl:default [Name or service not known]
Error fetching URL: https://goto.now/coBrJ
Error message: Cannot connect to host mdyfsvb.kwusg.my.id:443 ssl:default [Name or service not known]
Error fetching URL: https://ai-meta-support-request-review-956.ubpages.com/591-bolivia-id/
Error message: 404, message='Not Found', url='http://ai-meta-support-request-review-956.ubpages.com/591-bolivia-id/'
Error fetching URL: http://dimewayi.top/
Error message: Cannot connect to host

In [8]:
url_inputs = np.array(url_inputs)
adjacency_inputs = np.array(adjacency_inputs)
feature_inputs = np.array(feature_inputs)
labels = np.array(labels)

In [9]:
!pip install spektral

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 54.4 MB/s eta 0:00:00


In [10]:
from tensorflow.keras import models
from utils.layers import TransformerEncoder, PositionalEmbedding, GCN
from spektral.layers import GCNConv, GlobalSumPool

In [11]:
model = keras.models.load_model(
    'models/best_model(4).keras',
    custom_objects={'TransformerEncoder': TransformerEncoder,
                    'PositionalEmbedding': PositionalEmbedding,
                    'GCNConv': GCNConv,
                    'GlobalSumPool': GlobalSumPool,
                    'GCN': GCN})

In [13]:
model.evaluate([url_inputs, adjacency_inputs, feature_inputs], labels, verbose=1)

15/15 [==============================] - 7s 464ms/step - loss: 0.5196 - accuracy: 0.7810 - precision: 1.0000 - recall: 0.7810


[0.5195611119270325, 0.7809734344482422, 1.0, 0.7809734344482422]